# Phase 4: Computed Variables

Many OBR model variables have no direct ONS or EFO source — they're derived from other variables via arithmetic identities in the model equations file. This notebook:

1. **Parses** simple identity equations from `Macroeconomic_model_code_March_2025.txt`
2. **Loads** all available timeseries from the DB into a wide pandas DataFrame (quarters × variables)
3. **Evaluates** each parsed identity quarter-by-quarter using the DataFrame
4. **Stores** computed values back to the DB under source `COMPUTED`
5. **Reports** how many previously-missing variables are now covered

Only *simple* identities are computed here: equations of the form `VAR = expr` where `expr` contains only addition, subtraction, multiplication, division, constants, and variable references — **no lags** (`(-1)`), **no `@` functions**, **no `exp`/`log`/`^`**. Dynamic equations (which depend on lagged values or non-linear terms) require the full model solver and are out of scope.

In [1]:
import os, sys, re, sqlite3
import pandas as pd
import numpy as np

REPO      = os.path.expanduser('~/Documents/CBP/budget-master')
DB_PATH   = os.path.join(REPO, 'shaamini_tests', 'timeseries.db')
MODEL_TXT = os.path.join(REPO, 'docs', 'Macroeconomic_model_code_March_2025.txt')
PUB_DATE  = '2026-03'

sys.path.insert(0, REPO)

def db_coverage():
    conn  = sqlite3.connect(DB_PATH)
    total = conn.execute('SELECT COUNT(DISTINCT series_id) FROM observations').fetchone()[0]
    comp  = conn.execute("SELECT COUNT(DISTINCT series_id) FROM observations WHERE source='COMPUTED'").fetchone()[0]
    n_obs = conn.execute('SELECT COUNT(*) FROM observations').fetchone()[0]
    conn.close()
    return dict(with_data=total, computed=comp, total_obs=n_obs)

BASELINE = db_coverage()
print('Baseline:', BASELINE)

Baseline: {'with_data': 381, 'computed': 0, 'total_obs': 75085}


---
## Step 1 — Parse simple identity equations from the model code

The model code is EViews syntax. Each equation occupies one line (after stripping continuation). We keep only equations that:
- Are of the form `LHS = RHS` (not ratio equations like `X/X(-1) = ...`)
- Have no lags `(-1)`, no `@` functions, no `exp`/`log`/`^`
- LHS is a single clean variable name

The RHS is then converted to a Python expression by replacing variable names with `df['VAR']` references, allowing vectorised evaluation across all quarters at once.

In [2]:
VAR_RE = re.compile(r'\b([A-Z][A-Z0-9_]*)\b')

def parse_model_equations(filepath):
    """
    Read the model code and return a list of (lhs_var, rhs_expr, raw_line) tuples
    for simple arithmetic identities only.
    """
    identities = []
    skip_tokens = {'(-1)', '(-2)', '(-3)', '(-4)', '@', 'exp(', 'log(', 'sqrt('}

    with open(filepath) as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith('!'):
                continue

            # Skip dynamic / non-linear lines
            if any(tok in line for tok in skip_tokens):
                continue
            if '^' in line:
                continue

            # Must be VAR = RHS (not a ratio like VAR/VAR(-1) = ...)
            if '=' not in line:
                continue
            lhs_raw, _, rhs_raw = line.partition('=')
            lhs = lhs_raw.strip()
            rhs = rhs_raw.strip()

            # LHS must be a clean variable name (no operators, no slashes)
            if not re.fullmatch(r'[A-Z][A-Z0-9_]*', lhs):
                continue

            identities.append((lhs, rhs, line))

    return identities

equations = parse_model_equations(MODEL_TXT)
print(f'Simple identities found: {len(equations)}')
print('\nSample:')
for lhs, rhs, raw in equations[:10]:
    print(f'  {lhs:<12} = {rhs[:60]}')

Simple identities found: 219

Sample:
  CONSPS       = CONS  * PCE  / 100
  CDURPS       = (PCDUR  / 100)  * CDUR
  DINV         = (GDPM  + M  - SDE)  - CGG  - CONS  - VAL  - IF  - X
  DINVPS       = DINV  * PDINV  / 100
  DINVHH       = 0.07  * DINVPS
  DINVCG       = PSNI  - CGIPS  - LAIPS  - IPCPS  - IBPC  - (NPACG  + NPALA) 
  DP           = 1  / (1  + DISCO)  * ((DISCO  * FP  + SP)  / (DISCO  + SP))
  DV           = SV  / (DISCO  + SV)
  WB           = 0.31
  WP           = 0.54


---
## Step 2 — Load all available timeseries from the DB into a DataFrame

We build a wide DataFrame: rows = quarters (e.g. `2008Q1`), columns = model variable names. Each cell is the best available value for that variable in that quarter (ONS preferred over OBR; COMPUTED last).

Priority order: `ONS` > `OBR_EFO` > `OBR_EFO-annual` > `COMPUTED`

In [3]:
SOURCE_PRIORITY = {'ONS': 0, 'OBR_EFO': 1, '2026-03-annual': 2, 'COMPUTED': 3}

conn = sqlite3.connect(DB_PATH)

# Pull all observations — one row per (series_id, source, quarter, value)
raw = pd.read_sql_query(
    "SELECT series_id, source, quarter, value FROM observations ORDER BY quarter",
    conn
)
conn.close()

# Assign priority and keep best source per (variable, quarter)
raw['priority'] = raw['source'].map(lambda s: SOURCE_PRIORITY.get(s, 99))
raw = raw.sort_values('priority')
best = raw.drop_duplicates(subset=['series_id', 'quarter'], keep='first')

# Pivot to wide format
df = best.pivot(index='quarter', columns='series_id', values='value')
df = df.sort_index()  # sort quarters chronologically

print(f'DataFrame shape: {df.shape}  ({df.shape[0]} quarters × {df.shape[1]} variables)')
print(f'Quarter range  : {df.index[0]} → {df.index[-1]}')
print(f'Variables      : {list(df.columns[:10])} ...')

DataFrame shape: (341, 381)  (341 quarters × 381 variables)
Quarter range  : 1946Q1 → 2031Q1
Variables      : ['AAHH', 'AAROW', 'AL', 'ALAD', 'ALHH', 'APIIH', 'AVH', 'BANKROLL', 'BAROW', 'BBC'] ...


---
## Step 3 — Evaluate identities quarter-by-quarter

For each parsed equation we:
1. Convert the RHS EViews expression to a Python expression using `df['VAR']` substitution
2. Call `eval()` on it — pandas evaluates it across all quarters at once
3. Skip the equation if any input variable is missing from the DataFrame
4. Store the result as a new column in `df` so downstream equations can use it

We iterate multiple times (up to 5 passes) to handle chains where `A = B + C` and `D = A + E` — the second equation can only be computed once `A` is available.

In [4]:
def eviews_to_python(rhs, available_vars):
    """
    Convert EViews RHS expression to a Python expression for eval().
    Replaces VAR → df['VAR'] for known variable names.
    Returns None if any variable is missing from available_vars.
    """
    # Find all variable references in the RHS
    vars_in_rhs = set(VAR_RE.findall(rhs))
    # Remove numeric-like tokens (not variables)
    vars_in_rhs = {v for v in vars_in_rhs if not v.isdigit()}

    # Check all are available
    missing = vars_in_rhs - available_vars
    if missing:
        return None, missing

    # Replace VAR with df['VAR'] — longest names first to avoid partial matches
    py_expr = rhs
    for var in sorted(vars_in_rhs, key=len, reverse=True):
        py_expr = re.sub(rf'\b{re.escape(var)}\b', f"df['{var}']", py_expr)

    return py_expr, set()

computed_series = {}   # {var: pd.Series}
skipped         = {}   # {var: reason}
MAX_PASSES      = 6

pending = list(equations)

for pass_num in range(1, MAX_PASSES + 1):
    still_pending = []
    available = set(df.columns)

    for lhs, rhs, raw_line in pending:
        if lhs in df.columns:
            # Already have data for this variable — skip
            continue

        py_expr, missing = eviews_to_python(rhs, available)
        if py_expr is None:
            still_pending.append((lhs, rhs, raw_line))
            skipped[lhs] = f'missing inputs: {missing}'
            continue

        try:
            result = eval(py_expr)  # returns a pd.Series
            if hasattr(result, 'values'):
                df[lhs] = result
                computed_series[lhs] = result
            else:
                # Scalar constant (e.g. EUSUBP = 0)
                df[lhs] = float(result)
                computed_series[lhs] = pd.Series(float(result), index=df.index)
        except Exception as e:
            skipped[lhs] = f'eval error: {e}'

    pending = still_pending
    newly   = len(computed_series)
    print(f'Pass {pass_num}: computed {len(computed_series)} total, {len(pending)} still pending')
    if not pending:
        break

print(f'\nTotal computed : {len(computed_series)}')
print(f'Could not compute: {len(pending)} (missing inputs after all passes)')

Pass 1: computed 33 total, 67 still pending
Pass 2: computed 38 total, 61 still pending
Pass 3: computed 41 total, 58 still pending
Pass 4: computed 42 total, 57 still pending
Pass 5: computed 42 total, 57 still pending
Pass 6: computed 42 total, 57 still pending

Total computed : 42
Could not compute: 57 (missing inputs after all passes)


In [5]:
# Show what was computed
print('Variables computed this run:')
for var, series in sorted(computed_series.items()):
    n_valid = series.dropna().count() if hasattr(series, 'dropna') else len(df)
    print(f'  {var:<14} {n_valid} non-null quarters')

print('\nStill missing (first 20):')
for var, reason in list(skipped.items())[:20]:
    print(f'  {var:<14} {reason}')

Variables computed this run:
  AROW           156 non-null quarters
  EARN           136 non-null quarters
  EMPASC         156 non-null quarters
  FC             116 non-null quarters
  FYEMPMS        116 non-null quarters
  GDPM16         241 non-null quarters
  GDPMAL         285 non-null quarters
  GGFCD          305 non-null quarters
  GGNBCY         24 non-null quarters
  GGVAPS         116 non-null quarters
  LROW           156 non-null quarters
  MC             93 non-null quarters
  MCGG           305 non-null quarters
  MDINV          285 non-null quarters
  MGTFE          73 non-null quarters
  MIF            305 non-null quarters
  MINTY          73 non-null quarters
  MSGVA          1 non-null quarters
  MSGVAPS        116 non-null quarters
  MSGVAPSEMP     116 non-null quarters
  MSTFE          73 non-null quarters
  MTFE           73 non-null quarters
  MXG            117 non-null quarters
  MXS            117 non-null quarters
  NAFXLIC        156 non-null quarters
  NA

---
## Step 4 — Store computed values in the DB

Each computed series is stored with source `COMPUTED` and the same publication date as the OBR data. If the variable already has ONS or OBR_EFO data, the COMPUTED values coexist under the different source tag — they won't overwrite the authoritative data.

In [6]:
from cbp_fiscal_framework.db.timeseries_db import TimeSeriesDB

db   = TimeSeriesDB(DB_PATH)
conn = sqlite3.connect(DB_PATH)

stored = 0
for var, series in computed_series.items():
    # Convert to {quarter: value} dict, dropping NaNs
    if hasattr(series, 'dropna'):
        data = series.dropna().to_dict()
    else:
        data = {q: float(series) for q in df.index}

    if not data:
        continue

    # Ensure the series row exists
    label = conn.execute('SELECT label FROM series WHERE id=?', (var,)).fetchone()
    label = label[0] if label else var
    db.upsert_series(var, label, 'COMPUTED')
    db.upsert_observations(var, 'COMPUTED', PUB_DATE, data)
    stored += 1

conn.close()

after = db_coverage()
print(f'Variables stored         : {stored}')
print(f'New COMPUTED series      : +{after["computed"] - BASELINE["computed"]}')
print(f'Net new variables in DB  : +{after["with_data"] - BASELINE["with_data"]}')
print(f'New observations         : +{after["total_obs"] - BASELINE["total_obs"]:,}')

Variables stored         : 40
New COMPUTED series      : +40
Net new variables in DB  : +40
New observations         : +4,979


---
## Step 5 — Audit

How many of the ~500 model variables now have data from any source?

In [7]:
conn = sqlite3.connect(DB_PATH)

# Variables by source combination
print('=== OBSERVATIONS BY SOURCE ===')
for row in conn.execute("""
    SELECT source, COUNT(DISTINCT series_id) n_s, COUNT(*) n_o
    FROM observations GROUP BY source ORDER BY source
"""):
    print(f'  {str(row[0]):<24} series={row[1]:>4}  obs={row[2]:>8,}')

print()

# Still missing
no_data = conn.execute("""
    SELECT s.id, s.label, s.ons_code
    FROM series s
    WHERE s.id NOT IN (SELECT DISTINCT series_id FROM observations)
    ORDER BY s.id
""").fetchall()

has_code = [r for r in no_data if r[2]]
no_code  = [r for r in no_data if not r[2]]

print(f'Still no data: {len(no_data)} variables')
print(f'  Has ONS code (could fetch with new path): {len(has_code)}')
print(f'  No ONS code (dynamic / truly unresolvable): {len(no_code)}')

print('\nRemaining with ONS code (first 30):')
for r in has_code[:30]:
    print(f'  {r[0]:<14} ons={r[2]:<22} {r[1][:40]}')

conn.close()

=== OBSERVATIONS BY SOURCE ===
  COMPUTED                 series=  40  obs=   4,979
  OBR_EFO                  series= 130  obs=   9,148
  ONS                      series= 323  obs=  65,937

Still no data: 238 variables
  Has ONS code (could fetch with new path): 103
  No ONS code (dynamic / truly unresolvable): 135

Remaining with ONS code (first 30):
  AIC            ons=NKWX                   Stock of financial assets held by PNFCs
  ALROW          ons=-HBNR                  Total acquisition of UK claims on ROW (N
  BLIC           ons=NKZA                   Stock of bonds and Money Mkt instruments
  CDUR           ons=UTID                   HH final consumption expenditure: durabl
  CDURPS         ons=UTIB                   HH final consumption expenditure: durabl
  CGACADJ        ons=ANRT+ANRU+ANRV         Central Govt accruals adjustments
  CGCGLA         ons=QYJR                   Total grants from CG to LA
  CGISC          ons=M9WU+RUDY              CG imputed social contributi